In [ ]:
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption
import pytesseract
from PIL import Image
import os

# ====================== SETTINGS ======================
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False   # We will handle OCR ourselves with Tesseract

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

print("Starting conversion of Images.pdf...")
result = converter.convert(
    "../data/text_files/Images.pdf")  # Change path if needed
doc = result.document

print(f"\nTotal images found: {len(doc.pictures)}")
print("=" * 60)

# Create folder to save images
os.makedirs("extracted_images", exist_ok=True)

# ====================== PROCESS EACH IMAGE ======================
for i, picture in enumerate(doc.pictures):
    page_num = picture.prov[0].page_no if picture.prov else "Unknown"

    print(f"\n=== Image {i+1} (Page {page_num}) ===")

    if not picture.image:
        print("No image data found")
        continue

    # Save the image
    image_path = f"extracted_images/image_{i+1}_page_{page_num}.png"
    picture.image.save(image_path)
    print(f"Saved as → {image_path}")

    # ---------- Comments based on page type ----------
    if page_num == 1:
        # This is for Charts (Digital Graphics)
        print("Type: Digital Chart / Graph")
        print("→ No OCR needed")

    elif page_num == 2:
        # This is for Photographed images
        print("Type: Photograph")
        print("→ No OCR needed")

    elif page_num in [3, 4]:
        # This is for Invoices / Images with text
        print("Type: Invoice / Image with Text")
        print("→ Running OCR...")
        text = pytesseract.image_to_string(Image.open(image_path))
        print("----- Extracted Text -----")
        print(text.strip())
        print("--------------------------")

    else:
        # This is for Scanned documents
        print("Type: Scanned Document")
        print("→ Running OCR...")
        text = pytesseract.image_to_string(Image.open(image_path))
        print("----- Extracted Text -----")
        print(text.strip())
        print("--------------------------")

print("\nAll images processed successfully!")